In [19]:
import torch
from torch.utils.data import Dataset, DataLoader
from src.language_models.dictionary_corpus import Dictionary, Corpus, tokenize
from src.language_models.model import RNNModel as lstm
from src.language_models.utils import move_to_device, batchify, get_batch, repackage_hidden
import random
import pandas as pd
from collections import defaultdict
import numpy as np
import argparse
import re
import torch.nn as nn
import math
from pathlib import Path
import copy
from tqdm import tqdm

## Evaluation function 

In [3]:
device = torch.device("cpu")
eval_batch_size = 10
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')
val_data = batchify(corpus.valid, eval_batch_size, device)


In [57]:
ntokens = len(corpus.dictionary)

In [66]:
def evaluate(model, val_data, ntokens):
    # Turn on evaluation mode which disables dropout.
    model.eval()
    total_loss = 0
    hidden = move_to_device(model.init_hidden(eval_batch_size), device)
    with torch.no_grad():
        for i in range(0, val_data.size(0) - 1, 35):
            data, targets = get_batch(val_data, i, 35)
            data, targets = data.to(device), targets.to(device)
            
            
            output, hidden = model(data, hidden)
            output_flat = output.view(-1, ntokens)
            total_loss += (
                len(data) * nn.CrossEntropyLoss()(output_flat, targets).item()
            )
            del output, output_flat
            hidden = repackage_hidden(hidden)
            
    loss = total_loss / (len(val_data) - 1)
    
    return math.exp(loss)

## Extract relevant weight matrix from checkpoint, SV ablation, load the model with new weight matrix

In [67]:
def zero_out_lowest_singular_values(weight_matrix: torch.Tensor, n: int) -> torch.Tensor:
    # Compute SVD
    U, S, Vh = torch.linalg.svd(weight_matrix, full_matrices=False)
    
    # Zero out the n smallest singular values
    if n > 0:
        S[-n:] = 0
    
    # Reconstruct the matrix
    modified_weight = (U * S.unsqueeze(0)) @ Vh
    return modified_weight

In [68]:
def modify_checkpoint_weight(checkpoint_path, layer, weight_type, gate, n_singular_values_to_zero, device):
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model_weights = checkpoint["model_state_dict"]
    
    # Extract original weight tensor (concatenated gates)
    original_weight = model_weights[f'rnn.weight_{weight_type}_l{layer}']  # shape: (4*gate_dim, dim)
    
    # Split into gates
    gates = dict(zip(['input', 'forget', 'cell', 'output'], original_weight.chunk(4, dim=0)))
    
    # Modify specified gate's weight matrix
    modified_gate_weight = zero_out_lowest_singular_values(gates[gate], n_singular_values_to_zero)
    
    # Replace gate weight in gates dict
    gates[gate] = modified_gate_weight
    
    # Re-concatenate gates into full weight matrix
    modified_weight = torch.cat([gates[g] for g in ['input', 'forget', 'cell', 'output']], dim=0)
    
    # Create a copy of checkpoint to avoid modifying the original
    new_checkpoint = copy.deepcopy(checkpoint)
    new_checkpoint["model_state_dict"][f'rnn.weight_{weight_type}_l{layer}'] = modified_weight
    
    return new_checkpoint


# XP on small singuar values
1. Load checkpoint
2. evaluate model
3. extract relevant weight matrix
4. SVD on the matrix
5. ablate small singular values
6. test model
7. see whether perplexity remains within an epsilon (first try at 3%)

In [69]:
layers = [0,1]
weight_type = 'hh'
gates = ['cell','forget','input','output']

In [72]:
def evaluate_on_n_ablations(checkpoint_path, layers, weight_type, n):
    ablation ={}
    for layer in layers:
        ablation[f'layer_{layer}'] = {}
        for gate in gates:
            
            ablation[f'layer_{layer}'][gate] = []
            
            model = lstm('LSTM', ntokens, 650, 650, 2, 0, False).to(device)
            with open(checkpoint_path, "rb") as f:
                state_dict = torch.load(
                    f, map_location="cuda" if device == "cuda" else "cpu"
                )
                model.load_state_dict(state_dict["model_state_dict"])
                
            ppl_original = evaluate(model, val_data, ntokens)
            
            ablation[f'layer_{layer}'][gate]['original']=ppl_original
            
            for i in tqdm(range(n), desc=f"Layer {layer}, Gate {gate}"):
                check = modify_checkpoint_weight(checkpoint_path, layer, weight_type, gate, i+1, device)
                model = lstm('LSTM', ntokens, 650, 650, 2, 0, False).to(device)
                model.load_state_dict(check['model_state_dict'])
                ppl = evaluate(model, val_data, ntokens)
                ablation[f'layer_{layer}'][gate].append(ppl)
                
    return ablation   

In [73]:
checkpoint_path = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/epoch_40.pt'
evaluate_on_n_ablations(checkpoint_path, layers, weight_type, 100)

KeyboardInterrupt: 

# XP on large singular values
1. Load checkpoint
2. evaluate model
3. extract relevant weight matrix
4. SVD on the matrix
5. ablate large singular values
6. test model
7. see performance drops